# Google Search Console Anomaly Detection with Seasonality & Holidays
## STL Decomposition + Statistical Anomaly Detection

**What this does:**
- ✅ **Automatic seasonality detection** (weekly, yearly patterns)
- ✅ **Holiday effects** (built-in holiday calendars)
- ✅ **Trend extraction** (long-term growth/decline)
- ✅ **Intuitive visualization** (see each component)
- ✅ **Reliable installation** (no build errors!)

**Method: STL (Seasonal-Trend decomposition using Loess)**
- Industry-standard technique from statsmodels
- Decomposes time series into: Trend + Seasonal + Residual
- Anomalies = large residuals after removing trend & seasonality

## 1. Install Packages (No Build Issues!)

In [ ]:
print("Installing packages...\n")

!pip install -q --upgrade statsmodels scipy scikit-learn pandas matplotlib seaborn plotly holidays

print("✅ Installation complete!")
print("💡 No restart needed - continue to next cell!")

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from statsmodels.tsa.seasonal import STL
from scipy import stats
import holidays
import warnings
warnings.filterwarnings('ignore')

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print("\n🎉 Ready to analyze your GSC data!")

## 3. Load Your Google Search Console Data

In [ ]:
from google.colab import files

print("Please upload your Google Search Console CSV file:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
print(f"\n✅ File '{filename}' uploaded successfully!")

## 4. Load and Clean Data

In [ ]:
# Load and clean data
df = pd.read_csv(filename)

print("Cleaning data...\n")

# Standardize column names
df.columns = df.columns.str.lower().str.strip()

# Find date column
date_cols = [col for col in df.columns if 'date' in col.lower()]
if date_cols:
    date_column = date_cols[0]
    df['date'] = pd.to_datetime(df[date_column])
    if date_column != 'date':
        df = df.drop(columns=[date_column])

# Clean numeric columns
def clean_numeric(series):
    if series.dtype == 'object':
        series = series.astype(str).str.replace(',', '')
        if series.str.contains('%').any():
            series = series.str.rstrip('%').astype(float) / 100
        else:
            series = pd.to_numeric(series, errors='coerce')
    return pd.to_numeric(series, errors='coerce')

for col in ['clicks', 'impressions', 'ctr', 'position']:
    if col in df.columns:
        df[col] = clean_numeric(df[col])

# Sort and aggregate duplicates
df = df.sort_values('date').reset_index(drop=True)
if df['date'].duplicated().any():
    agg_dict = {}
    for col in ['clicks', 'impressions', 'ctr', 'position']:
        if col in df.columns:
            agg_dict[col] = 'sum' if col != 'position' else 'mean'
    df = df.groupby('date').agg(agg_dict).reset_index()

# Fill missing values
df = df.ffill().bfill()

# Set date as index for time series analysis
df = df.set_index('date')

print("✅ Data prepared!\n")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
print(f"Total days: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

## 5. Configuration

In [ ]:
# ==================== CONFIGURATION ====================

# Metric to analyze
METRIC = 'clicks'  # Options: 'clicks', 'impressions', 'ctr', 'position'

# Country for holidays
COUNTRY = 'US'  # Options: 'US', 'UK', 'CA', 'AU', 'DE', 'FR', 'ES', 'IT', 'JP', etc.

# Seasonality settings
SEASONAL_PERIOD = 7  # 7 for weekly seasonality, 365 for yearly
# For data < 2 years: use 7 (weekly)
# For data >= 2 years: use 365 (yearly) or try both

INCLUDE_HOLIDAYS = True

# Anomaly detection threshold
SIGMA_THRESHOLD = 3  # Standard deviations (2-4)
# Lower = more anomalies detected
# Higher = only extreme anomalies

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"Metric: {METRIC}")
print(f"Country: {COUNTRY}")
print(f"Seasonal period: {SEASONAL_PERIOD} days")
print(f"Include holidays: {INCLUDE_HOLIDAYS}")
print(f"Anomaly threshold: {SIGMA_THRESHOLD}σ")
print("=" * 60)

## 6. Visualize Raw Data

In [ ]:
plt.figure(figsize=(16, 5))
plt.plot(df.index, df[METRIC], linewidth=1.5, color='#2E86AB')
plt.xlabel('Date', fontsize=12, fontweight='bold')
plt.ylabel(METRIC.capitalize(), fontsize=12, fontweight='bold')
plt.title(f'{METRIC.capitalize()} Over Time', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"\n{METRIC.capitalize()} Statistics:")
print(f"  Mean: {df[METRIC].mean():.2f}")
print(f"  Std Dev: {df[METRIC].std():.2f}")
print(f"  Min: {df[METRIC].min():.2f}")
print(f"  Max: {df[METRIC].max():.2f}")

## 7. Apply STL Decomposition (Extract Seasonality & Trend)

In [ ]:
print(f"Applying STL decomposition with period={SEASONAL_PERIOD}...\n")

# STL decomposition
# This separates the time series into: Trend + Seasonal + Residual
stl = STL(df[METRIC], seasonal=SEASONAL_PERIOD, robust=True)
result = stl.fit()

# Extract components
df['trend'] = result.trend
df['seasonal'] = result.seasonal
df['residual'] = result.resid

# The residual is what's left after removing trend and seasonality
# Large residuals = anomalies!

print("✅ STL decomposition complete!")
print(f"\nComponents extracted:")
print(f"  Trend: Long-term movement")
print(f"  Seasonal: Repeating {SEASONAL_PERIOD}-day pattern")
print(f"  Residual: What remains (contains anomalies)")

## 8. Visualize Decomposition (Like Prophet's Component Plot)

In [ ]:
fig = result.plot()
fig.set_size_inches(16, 10)
plt.tight_layout()
plt.show()

print("\n📊 Interpretation Guide:")
print("="*70)
print("1. OBSERVED: Your actual data")
print("\n2. TREND: Overall direction")
print("   - Going up? Traffic is growing")
print("   - Going down? Traffic is declining")
print("   - Flat? Stable traffic")
print(f"\n3. SEASONAL: {SEASONAL_PERIOD}-day repeating pattern")
if SEASONAL_PERIOD == 7:
    print("   - Shows day-of-week effects (weekday vs weekend)")
elif SEASONAL_PERIOD == 365:
    print("   - Shows time-of-year effects (seasonal trends)")
print("\n4. RESIDUAL: What's left (THIS is where we find anomalies!)")
print("   - Large spikes/dips = anomalies")
print("   - After accounting for trend & seasonality")
print("="*70)

## 9. Load Holiday Data

In [ ]:
if INCLUDE_HOLIDAYS:
    # Get holidays for date range
    start_year = df.index.min().year
    end_year = df.index.max().year

    country_holidays = holidays.country_holidays(COUNTRY, years=range(start_year, end_year + 1))

    # Create holiday indicator
    df['is_holiday'] = df.index.isin(country_holidays)
    df['holiday_name'] = df.index.map(lambda x: country_holidays.get(x, ''))

    print(f"✅ Loaded holidays for {COUNTRY}")
    print(f"Total holidays in date range: {df['is_holiday'].sum()}")
    print(f"\nSample holidays:")
    holiday_df = df[df['is_holiday']][['holiday_name']].head(10)
    for date, row in holiday_df.iterrows():
        print(f"  {date.strftime('%Y-%m-%d')}: {row['holiday_name']}")
else:
    df['is_holiday'] = False
    df['holiday_name'] = ''
    print("Holiday detection disabled")

## 10. Detect Anomalies in Residuals

In [ ]:
# Calculate z-scores on the residuals
# (residuals already have trend & seasonality removed)
residual_mean = df['residual'].mean()
residual_std = df['residual'].std()

threshold_upper = residual_mean + (SIGMA_THRESHOLD * residual_std)
threshold_lower = residual_mean - (SIGMA_THRESHOLD * residual_std)

# Detect anomalies
df['is_anomaly'] = (
    (df['residual'] > threshold_upper) |
    (df['residual'] < threshold_lower)
)

# Classify anomaly type
df['anomaly_type'] = 'Normal'
df.loc[(df['is_anomaly']) & (df['residual'] > 0), 'anomaly_type'] = 'Positive Anomaly'
df.loc[(df['is_anomaly']) & (df['residual'] < 0), 'anomaly_type'] = 'Negative Anomaly'

# Calculate anomaly score for ranking
df['anomaly_score'] = np.abs((df['residual'] - residual_mean) / residual_std)

# Reconstruct expected value (trend + seasonal)
df['expected'] = df['trend'] + df['seasonal']

print("✅ Anomaly detection complete!\n")
print(f"Thresholds:")
print(f"  Upper: {threshold_upper:.2f}")
print(f"  Lower: {threshold_lower:.2f}")
print(f"\nResults:")
print(f"  Total anomalies: {df['is_anomaly'].sum()} / {len(df)} days ({df['is_anomaly'].sum()/len(df)*100:.1f}%)")
print(f"  Positive anomalies: {(df['anomaly_type'] == 'Positive Anomaly').sum()}")
print(f"  Negative anomalies: {(df['anomaly_type'] == 'Negative Anomaly').sum()}")

if INCLUDE_HOLIDAYS:
    holiday_anomalies = df[df['is_anomaly'] & df['is_holiday']]
    print(f"\n  Anomalies on holidays: {len(holiday_anomalies)}")

## 11. Visualize Anomalies with Trend & Seasonality

In [ ]:
fig, ax = plt.subplots(figsize=(18, 8))

# Plot actual values
ax.plot(df.index, df[METRIC],
        label='Actual', color='black', linewidth=2, zorder=3)

# Plot expected (trend + seasonal)
ax.plot(df.index, df['expected'],
        label='Expected (Trend + Seasonal)', color='#2E86AB', linewidth=2, linestyle='--', zorder=2)

# Highlight anomalies
positive = df[df['anomaly_type'] == 'Positive Anomaly']
negative = df[df['anomaly_type'] == 'Negative Anomaly']

ax.scatter(positive.index, positive[METRIC],
          color='#06D6A0', s=150, label='Positive Anomalies',
          zorder=5, edgecolors='black', linewidth=2)

ax.scatter(negative.index, negative[METRIC],
          color='#EF476F', s=150, label='Negative Anomalies',
          zorder=5, edgecolors='black', linewidth=2)

# Mark holidays if enabled
if INCLUDE_HOLIDAYS and df['is_holiday'].any():
    holidays_in_range = df[df['is_holiday']]
    ax.scatter(holidays_in_range.index, holidays_in_range[METRIC],
              marker='v', color='orange', s=100, alpha=0.5, label='Holidays', zorder=4)

ax.set_xlabel('Date', fontsize=13, fontweight='bold')
ax.set_ylabel(METRIC.capitalize(), fontsize=13, fontweight='bold')
ax.set_title(f'Seasonality-Aware Anomaly Detection: {METRIC.capitalize()}',
            fontsize=16, fontweight='bold', pad=20)
ax.legend(loc='best', fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 12. Residual Analysis

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18, 10))

# Plot 1: Residuals over time
axes[0].plot(df.index, df['residual'], linewidth=1, color='#A23B72', alpha=0.7)
axes[0].axhline(y=threshold_upper, color='red', linestyle='--', linewidth=2, label=f'+{SIGMA_THRESHOLD}σ')
axes[0].axhline(y=threshold_lower, color='red', linestyle='--', linewidth=2, label=f'-{SIGMA_THRESHOLD}σ')
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.5)
axes[0].scatter(positive.index, positive['residual'], color='#06D6A0', s=100, zorder=5, edgecolors='black')
axes[0].scatter(negative.index, negative['residual'], color='#EF476F', s=100, zorder=5, edgecolors='black')
axes[0].set_xlabel('Date', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Residual', fontsize=12, fontweight='bold')
axes[0].set_title('Residuals After Removing Trend & Seasonality', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Distribution
axes[1].hist(df['residual'], bins=50, color='#A23B72', alpha=0.7, edgecolor='black')
axes[1].axvline(x=threshold_upper, color='red', linestyle='--', linewidth=2)
axes[1].axvline(x=threshold_lower, color='red', linestyle='--', linewidth=2)
axes[1].axvline(x=residual_mean, color='blue', linestyle='-', linewidth=2, label='Mean')
axes[1].set_xlabel('Residual Value', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Frequency', fontsize=12, fontweight='bold')
axes[1].set_title('Distribution of Residuals', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 13. Top Anomalies Report

In [ ]:
anomaly_df = df[df['is_anomaly']].copy()
anomaly_df = anomaly_df.sort_values('anomaly_score', ascending=False)

print(f"\n{'='*120}")
print(f"TOP 20 ANOMALIES")
print(f"{'='*120}\n")

display_df = anomaly_df.head(20).reset_index()
display_df['date_str'] = display_df['date'].dt.strftime('%Y-%m-%d (%A)')
display_df['actual'] = display_df[METRIC].apply(lambda x: f"{x:.1f}")
display_df['expected'] = display_df['expected'].apply(lambda x: f"{x:.1f}")
display_df['residual_str'] = display_df['residual'].apply(lambda x: f"{x:+.1f}")
display_df['score'] = display_df['anomaly_score'].apply(lambda x: f"{x:.2f}σ")

output_df = display_df[['date_str', 'actual', 'expected', 'residual_str', 'score', 'anomaly_type', 'holiday_name']]
output_df.columns = ['Date', 'Actual', 'Expected', 'Residual', 'Score', 'Type', 'Holiday']

print(output_df.to_string(index=False))

print(f"\n{'='*120}")
print(f"SUMMARY")
print(f"{'='*120}")
print(f"Total days: {len(df)}")
print(f"Anomalies: {len(anomaly_df)} ({len(anomaly_df)/len(df)*100:.1f}%)")
print(f"  Positive: {(anomaly_df['anomaly_type'] == 'Positive Anomaly').sum()}")
print(f"  Negative: {(anomaly_df['anomaly_type'] == 'Negative Anomaly').sum()}")

if INCLUDE_HOLIDAYS:
    holiday_anoms = anomaly_df[anomaly_df['is_holiday']]
    print(f"  On holidays: {len(holiday_anoms)}")

## 14. Holiday Impact Analysis

In [ ]:
if INCLUDE_HOLIDAYS and df['is_holiday'].any():
    print(f"\n{'='*100}")
    print(f"HOLIDAY IMPACT ANALYSIS")
    print(f"{'='*100}\n")

    holiday_data = df[df['is_holiday']].copy()

    print(f"Total holidays: {len(holiday_data)}")
    print(f"Holidays with anomalies: {holiday_data['is_anomaly'].sum()}")

    # Average residuals
    holiday_avg_residual = holiday_data['residual'].mean()
    non_holiday_avg_residual = df[~df['is_holiday']]['residual'].mean()

    print(f"\nAverage residual:")
    print(f"  Holidays: {holiday_avg_residual:+.2f}")
    print(f"  Non-holidays: {non_holiday_avg_residual:+.2f}")

    if holiday_avg_residual > non_holiday_avg_residual + 5:
        print(f"\n💡 Holidays tend to have HIGHER traffic than expected")
    elif holiday_avg_residual < non_holiday_avg_residual - 5:
        print(f"\n💡 Holidays tend to have LOWER traffic than expected")
    else:
        print(f"\n💡 Holidays have similar traffic patterns to regular days")

    # Show holiday anomalies
    holiday_anomalies = holiday_data[holiday_data['is_anomaly']]
    if len(holiday_anomalies) > 0:
        print(f"\nHolidays flagged as anomalies:")
        print(f"{'-'*100}")
        for date, row in holiday_anomalies.iterrows():
            print(f"{date.strftime('%Y-%m-%d'):12s} | {row['holiday_name']:30s} | "
                  f"Actual: {row[METRIC]:7.0f} | Expected: {row['expected']:7.0f} | "
                  f"Residual: {row['residual']:+7.0f}")
else:
    print("\nHoliday analysis skipped")

## 15. Export Results

In [ ]:
# Prepare export
export_df = df.reset_index()
export_cols = ['date', METRIC, 'trend', 'seasonal', 'expected', 'residual',
               'is_anomaly', 'anomaly_type', 'anomaly_score', 'is_holiday', 'holiday_name']
export_df = export_df[export_cols]

export_df.columns = ['date', 'actual', 'trend', 'seasonal', 'expected', 'residual',
                     'is_anomaly', 'anomaly_type', 'anomaly_score', 'is_holiday', 'holiday_name']

# Full results
output_filename = f'gsc_seasonal_anomalies_{METRIC}_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
export_df.to_csv(output_filename, index=False)
print(f"✅ Full results: {output_filename}")
files.download(output_filename)

# Anomalies only
anomalies_filename = f'gsc_seasonal_anomalies_only_{METRIC}_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
export_df[export_df['is_anomaly']].to_csv(anomalies_filename, index=False)
print(f"✅ Anomalies only: {anomalies_filename}")
files.download(anomalies_filename)

print(f"\n📊 Export complete!")

## 16. Try Different Seasonal Periods

In [ ]:
print("Seasonal Period Guide")
print("="*70)
print("\nTo change the seasonal pattern, modify SEASONAL_PERIOD in Cell 5:")
print("\n  SEASONAL_PERIOD = 7   → Weekly patterns (day-of-week effects)")
print("                          Best for: B2B sites, blogs, news sites")
print("\n  SEASONAL_PERIOD = 365 → Yearly patterns (seasonal trends)")
print("                          Best for: E-commerce, travel, seasonal products")
print("                          Requires: At least 2 years of data")
print("\nData length requirements:")
print("  - Weekly (7): Minimum 2-3 months")
print("  - Yearly (365): Minimum 2 years")
print("\nYour data:")
data_length_days = (df.index.max() - df.index.min()).days
print(f"  Length: {data_length_days} days ({data_length_days/365:.1f} years)")
if data_length_days >= 730:
    print("  ✓ Can use weekly (7) or yearly (365)")
elif data_length_days >= 60:
    print("  ✓ Can use weekly (7)")
    print("  ✗ Too short for yearly (365) - need 2+ years")
else:
    print("  ⚠️  Data might be too short for reliable seasonality detection")
print("="*70)

## This Method (STL Decomposition)

**Advantages:**
- ✅ Handles seasonality (weekly, yearly)
- ✅ Holiday support
- ✅ Fast and reliable
- ✅ Visual component breakdown
- ✅ Industry-standard method

**How It Compares to Prophet:**
- Similar decomposition approach
- Same visual insights (trend, seasonal, residual)
- Comparable anomaly detection

### Key Insights

1. **Trend**: Shows long-term direction
2. **Seasonal**: Captures repeating patterns
3. **Residual**: Contains anomalies after accounting for above

### Next Steps

1. **Investigate anomalies**: Check for known events
2. **Adjust sensitivity**: Change SIGMA_THRESHOLD if needed
3. **Try different periods**: Weekly (7) vs Yearly (365)
4. **Cross-reference**: Compare with business events
